Learnt from
https://www.kaggle.com/code/uysimty/keras-cnn-dog-or-cat-classification

# Import data from Kaggle

In [ ]:
from google.colab import files
files.upload()
!rm -r ~/.kaggle
!mkdir ~/.kaggle
!mv ./kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c dogs-vs-cats

# Import libraries

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from keras.preprocessing.image import load_img
from keras.utils import to_categorical, image_dataset_from_directory
import matplotlib.pyplot as plt
import random
import os

from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/0 cnn training/data'
train_path = data_path+'/train/train/'
test_path = data_path + '/test1/test1/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Define constants


In [ ]:
IMG_WIDTH = 128
IMG_HEIGHT = 128
CHANNEL = 3 # BGR
BATCH_SIZE = 32
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, CHANNEL)

# Prepare data via tensorflow dataset


In [ ]:
# almost faithful recreation from link
def augment(image):
  # rotate
  image = tf.image.rat90(image, k=tf.random.uniform(shape=[],minval=0,maxval=4,dtype=tf.int32))

  # shear range
  image = tf.keras.preprocessing.image.random_shear(image,intensity=0.2)

  # zoom range
  image = tf.keras.preprocessing.image.random_zoom(image,zoom_range=(0.8,1.2))

  # horizontal flip
  image = tf.image.random_flip_left_right(image)

  # width shift range
  image = tf.keras.preprocessing.image.random_shift(image,wrg=0.1,hrg=0,row_axis=0,col_axis=1)

  # height shift range
  image = tf.keras.preprocessing.image.random_shift(image,wrg=0,hrg=0.1,row_axis=1,col_axis=0)

  return image

In [ ]:
train_dataset = image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='training',
    batch_size=BATCH_SIZE,
    image_size=IMG_SHAPE[:2],
    seed = 128
)
train_dataset = train_dataset.map(augment)
train_size = tf.data.experimental.cardinality(train_dataset).numpy()

val_dataset = image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='validation',
    batch_size=BATCH_SIZE,
    image_size=IMG_SHAPE[:2],
    seed = 128
)
val_dataset = val_dataset.map(augment)
val_size = tf.data.experimental.cardinality(val_dataset).numpy()

Found 0 files belonging to 0 classes.
Using 0 files for training.


ValueError: ignored

# Build model


1. **Input layer**
- Converts the image into a ((w * h * c),1) array
2. **Feature extraction layer**
- conv layer - extract features by mask
- batch normalization - normalize activation of previous layers; current input
(https://keras.io/api/layers/normalization_layers/batch_normalization/)
- pooling layer - either average or max pooling
- drop out - to reduce dependence of weightage connection; reduce overfitting
3. **Fully connected layer**
- Connects the hidden layers to output layer, flattening the array into a 1D array
4. **Output layer**

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout, Dense, Flatten, Activation, BatchNormalization

model = Sequential()

# Layer 1 - Input layer to Hidden 1
model.add(Conv2D(32,(3,3), activation='relu', input_shape=IMG_SHAPE)) # Input layer
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Layer 2 - Hidden 1 to Hidden 2
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Layer 3 - Hidden 2 to Hidden 3
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Layer 4 - Hidden 3 to Fully connected layer
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Layer 5 - Output layer
model.add(Dense(2, activation='softmax')) # 2 for cat, dog
model.compile(loss='categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 126, 32)      896       
                                                                 
 batch_normalization (Batch  (None, 126, 126, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 63, 63, 32)        0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 63, 63, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 61, 32)        9248      
                                                                 
 batch_normalization_1 (Bat  (None, 61, 61, 32)        1

# Callback

- To customize the behaviour of training.
- Callbacks are used to perform specific actions at various points during training.
- Also provides a way to monitor and control training process dynamically

Common callbacks:
1. **Model Checkpoint**
- Save weights at specified intervals during training, allow saving the best model or continue training from last checkpoint
2. **Early Stopping**
- Monitor specified metric (e.g., validation loss) and stop training if metric doesn't improve after a certain number of epochs, help prevent overfitting
3. **Learning Rate Scheduler**
- Adjust learning rate during training based on a predefined schedule or custom function
4. **Tensorboard**
- Visualization of training metrics and model graphs
5. **Reduce LR on Plateau**
- reduce learning rate if monitored metric reaches a plateau, allow fine-tuning of model convergence

Example given from link,
1. Stop learning after 10 epochs and val_loss value does not decrease
2. Reduce learning rate when accuracy does not increase for 2 epochs

In [ ]:
from keras.src.callbacks import EarlyStopping
earlystop = EarlyStopping(patience=10)

In [ ]:
from keras.src.callbacks import ReduceLROnPlateau
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc',
                                            patience=2,
                                            verbose=1,
                                            factor=0.5,
                                            min_lr= 0.00001)

In [ ]:
callbacks = [earlystop, learning_rate_reduction]

# Fit model

In [ ]:
epochs = 10
history = model.fit(
    train_dataset/255,
    epochs = epochs,
    validation_data = val_dataset/255,
    validation_steps = val_size//BATCH_SIZE,
    steps_per_epoch = train_size//BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
model.save_weights('cat-dog-model.h5')

# Visualise training

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1,figsize=(12,12))
ax1.plot(history.history['loss'],color='b',label='Training loss')
ax1.plot(history.history['val_loss'],color='b',label='Validation loss')
ax1.set_xticks(np.arrange(1,epochs,1))
ax1.set_yticks(np.arrange(0,1,0.1))

ax2.plot(history.history['acc'],color='b',label='Training accuracy')
ax2.plot(history.history['val_acc'],color='b',label='Validation accuracy')
ax2.set_xticks(np.arrange(1,epochs,1))

legend = plt.legend(loc='best',shadow=True)
plt.tight_layout()
plt.show()

# Visualise kernel


In [ ]:
# Retrieve the kernel weights
kernel_weights = history.layers[0].get_weights()[0]

# Visualize each kernel
fig, axes = plt.subplots(nrows = 10, ncols = 10, figsize=(10,20))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(kernel_weights[:, :, 0, i], cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Prepare test data

In [ ]:
test_dataset = image_dataset_from_directory(
    test_path,
    batch_size=BATCH_SIZE,
    image_size=IMG_SHAPE[:2],
    seed = 128
)

test_size = tf.data.experimental.cardinality(test_dataset).numpy()

In [ ]:
import pandas as pd
test_filenames = os.listdir(data_path+'/test1/test1')
test_df = pd.DataFrame({
    'filename':test_filenames
})

# Predict

In [ ]:
predict = model.predict(test_dataset/255, steps=np.ceil(test_size/BATCH_SIZE))
print(predict[:10])

In [ ]:
test_df['category']=np.argmax(predict,axis=1)